In [1]:
import pandas as pd
import numpy as np
import joblib

# Load processed data
df = pd.read_csv("../data/processed/features_crudop_logs.csv")

# Load trained model bundle
bundle = joblib.load("../models/v1/model_bundle.pkl")

models = bundle["models"]
features = bundle["feature_map"]
targets = bundle["targets"]

In [2]:
df_pred = df.copy()

for t in targets:
    feat = features[t]
    df_pred[f"{t}_pred"] = models[t].predict(df[feat])

In [3]:
for t in targets:
    df_pred[f"{t}_error"] = abs(df_pred[t] - df_pred[f"{t}_pred"])
    df_pred[f"{t}_ratio"] = df_pred[f"{t}_error"] / (df_pred[f"{t}_pred"] + 1e-6)

In [4]:
df_pred.columns

Index(['success_vol', 'success_rt_avg', 'fail_vol', 'fail_rt_avg', 'hour',
       'hour_sin', 'hour_cos', 'operation_DELETE', 'operation_GET',
       'operation_POST', 'operation_PUT', 'total_vol', 'fail_ratio',
       'latency_gap', 'vol_per_latency', 'success_vol_pred', 'fail_vol_pred',
       'success_rt_avg_pred', 'fail_rt_avg_pred', 'success_vol_error',
       'success_vol_ratio', 'fail_vol_error', 'fail_vol_ratio',
       'success_rt_avg_error', 'success_rt_avg_ratio', 'fail_rt_avg_error',
       'fail_rt_avg_ratio'],
      dtype='object')

In [5]:
df_pred["operation"] = df_pred[
    ["operation_GET", "operation_POST", "operation_PUT", "operation_DELETE"]
].idxmax(axis=1)

df_pred["operation"] = df_pred["operation"].str.replace("operation_", "")

df_pred["operation"].head()

0       GET
1      POST
2       PUT
3    DELETE
4       GET
Name: operation, dtype: object

In [6]:
thresholds = {}

grouped = df_pred.groupby(["operation", "hour"])

for (op, hour), group in grouped:
    
    if op not in thresholds:
        thresholds[op] = {}
    
    if hour not in thresholds[op]:
        thresholds[op][hour] = {}
    
    for t in targets:
        percent_thr = group[f"{t}_ratio"].quantile(0.999)
        abs_thr = group[f"{t}_error"].quantile(0.999)
        
        thresholds[op][hour][t] = {
            "percent_threshold": percent_thr,
            "abs_threshold": abs_thr
        }

In [9]:
bundle["thresholds"] = thresholds

import joblib
joblib.dump(bundle, "../models/v1/model_bundle.pkl")

print("✅ Thresholds saved")

✅ Thresholds saved
